# LLM Criticism and Human Correction

This notebook prepares Claude Code critic batches for the ACT-inspired review stage, parses critic outputs, and creates a correction sheet for human adjudication.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
LLM_DIR = CLASSIFICATION_DIR / "llm_annotation"
CRITIC_DIR = LLM_DIR / "critic_batches"
CRITIC_RAW_DIR = LLM_DIR / "critic_raw_outputs"
CORRECTION_DIR = CLASSIFICATION_DIR / "human_correction"

for path in [CRITIC_DIR, CRITIC_RAW_DIR, CORRECTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

ANNOTATOR_LABELS_PATH = LLM_DIR / "annotator_parsed/frame_llm_annotator_labels.csv"
LLM_POOL_PATH = LLM_DIR / "frame_llm_training_pool_with_annotation_ids.csv"
CRITIC_PROMPT_PATH = PROJECT_ROOT / "notebooks/01_classification/prompts/critic_v1.md"
BATCH_SIZE = 75

## Prepare Critic Batches

In [ ]:
if not ANNOTATOR_LABELS_PATH.exists():
    print(f"No parsed annotator labels found yet: {ANNOTATOR_LABELS_PATH.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Run LLM annotation and parsing first.")

pool = pd.read_csv(LLM_POOL_PATH)
labels = pd.read_csv(ANNOTATOR_LABELS_PATH)
critic_input = labels.merge(
    pool[["annotation_id", "context_id", "analysis_unit", "raw_form", "target_sentence_plus_adjacent"]],
    on=["annotation_id"],
    how="left",
    suffixes=("", "_pool"),
)

batch_rows = []
for batch_index, start in enumerate(range(0, len(critic_input), BATCH_SIZE), start=1):
    batch = critic_input.iloc[start : start + BATCH_SIZE].copy()
    batch_name = f"critic_batch_{batch_index:03d}.jsonl"
    batch_path = CRITIC_DIR / batch_name
    with batch_path.open("w", encoding="utf-8") as handle:
        for _, row in batch.iterrows():
            payload = {
                "annotation_id": row["annotation_id"],
                "target": row.get("analysis_unit", ""),
                "raw_form": row.get("raw_form", ""),
                "passage": row.get("target_sentence_plus_adjacent", ""),
                "proposed_clinical_frame_present": row.get("clinical_frame_present"),
                "proposed_lived_experience_frame_present": row.get("lived_experience_frame_present"),
                "proposed_clinical_evidence": row.get("clinical_evidence", ""),
                "proposed_lived_evidence": row.get("lived_evidence", ""),
                "proposed_uncertainty_note": row.get("uncertainty_note", ""),
            }
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
    batch_rows.append({"batch_name": batch_name, "rows": len(batch)})

manifest = pd.DataFrame(batch_rows)
manifest_path = LLM_DIR / "critic_batch_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print(f"Prompt: {CRITIC_PROMPT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {len(manifest):,} critic batches to {CRITIC_DIR.relative_to(PROJECT_ROOT)}")

## Parse Critic Outputs and Create Correction Sheet

The review threshold is not fixed here. It should be calibrated from pilot behaviour and adjusted if the critic flags an implausibly large share of rows.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue
            try:
                rows.append(json.loads(text))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path.name} line {line_number}: {error}") from error
    return rows

raw_files = sorted(CRITIC_RAW_DIR.glob("*.jsonl"))
if not raw_files:
    print(f"No critic outputs found in {CRITIC_RAW_DIR.relative_to(PROJECT_ROOT)} yet.")
else:
    critic_rows = []
    for path in raw_files:
        for row in read_jsonl(path):
            row["source_file"] = path.name
            critic_rows.append(row)
    critic = pd.DataFrame(critic_rows)
    critic_path = LLM_DIR / "frame_llm_critic_scores.csv"
    critic.to_csv(critic_path, index=False)

    labels = pd.read_csv(ANNOTATOR_LABELS_PATH)
    correction = labels.merge(critic, on="annotation_id", how="left", suffixes=("_annotator", "_critic"))
    correction = correction.merge(
        pool[["annotation_id", "context_id", "analysis_unit", "lsc_year", "raw_form", "target_sentence_plus_adjacent"]],
        on="annotation_id",
        how="left",
    )
    correction["max_axis_error_prob"] = correction[["clinical_error_prob", "lived_error_prob"]].max(axis=1)
    correction = correction.sort_values("max_axis_error_prob", ascending=False)
    correction["corrected_clinical_frame_present"] = ""
    correction["corrected_lived_experience_frame_present"] = ""
    correction["correction_note"] = ""
    correction_path = CORRECTION_DIR / "frame_llm_correction_sheet.csv"
    correction.to_csv(correction_path, index=False)
    print(f"Wrote critic scores: {critic_path.relative_to(PROJECT_ROOT)}")
    print(f"Wrote human correction sheet: {correction_path.relative_to(PROJECT_ROOT)}")